# 06 — Extra experiments for the IEEE Access revision

| Part | What it produces | Review item | Needs GPU time |
|---|---|---|---|
| A | Train / val / test image counts per class | B11 | seconds |
| B | Bootstrap 95% CIs, validation-vs-test gap, per-model threshold tuning on validation | B1, B2, B5 | minutes |
| C | McNemar for all 6 pairs with Holm correction + disagreement / oracle / ensemble analysis | B3 | minutes |
| D | Robustness on the **full test set**, 3 severities per degradation, with CIs | B4 | ~20–40 min |
| E | Duplicate / near-duplicate check (pHash + MD5) train/val vs test | B5 | ~10–20 min |
| F | Inference timing protocol (warm-up, sync, batch 1 and 32, FP32 and AMP) | B7 | minutes |
| G | Quantitative Grad-CAM concentration | B8 | minutes |
| H | **Extra seeds** (train again with new seeds) + mean ± SD table | B2 | **hours** (like notebooks 01–04, per seed) |
| I | Zips all tables for you to send back | | seconds |



## 0. Environment

In [ ]:
import importlib, subprocess, sys, os

def ensure(pkg, import_name=None):
    try:
        importlib.import_module(import_name or pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

for pkg, imp in [("timm", "timm"), ("opencv-python", "cv2"), ("scikit-learn", "sklearn"),
                 ("statsmodels", "statsmodels"), ("imagehash", "imagehash"), ("seaborn", "seaborn")]:
    ensure(pkg, imp)

import random, json, glob, io, time, hashlib, shutil, math
import numpy as np, pandas as pd
import torch, torchvision, timm, cv2
from PIL import Image
import torch.nn as nn
import torchvision.models as tvm
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm.auto import tqdm

print("PyTorch", torch.__version__, "| torchvision", torchvision.__version__, "| timm", timm.__version__)
print("CUDA available:", torch.cuda.is_available(),
      "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 0b. Get the dataset onto Colab's local disk

In [ ]:
import glob
DATA_ROOT = "/content/deepfake_data"
ZIP_IN_DRIVE = "/content/drive/MyDrive/deepfake-and-real-images.zip"

def dataset_ready():
    return bool(glob.glob(DATA_ROOT + "/**/Test", recursive=True)) or bool(glob.glob(DATA_ROOT + "/**/test", recursive=True))

if dataset_ready():
    print("Dataset already present in", DATA_ROOT)
else:
    os.makedirs(DATA_ROOT, exist_ok=True)
    if os.path.exists(ZIP_IN_DRIVE):
        print("Unzipping the zip found in Drive ...")
        subprocess.run(["unzip", "-q", "-o", ZIP_IN_DRIVE, "-d", DATA_ROOT], check=True)
    elif os.path.exists("/content/kaggle.json") or os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")):
        print("Downloading from Kaggle ...")
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        if os.path.exists("/content/kaggle.json"):
            shutil.copy("/content/kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
        os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
        ensure("kaggle")
        subprocess.run(["kaggle", "datasets", "download", "-d", "manjilkarki/deepfake-and-real-images",
                        "-p", DATA_ROOT, "--unzip"], check=True)
    else:
        raise FileNotFoundError("No dataset found. Upload kaggle.json to /content (left sidebar -> upload) "
                                "or put deepfake-and-real-images.zip in your Drive's MyDrive, then re-run this cell.")
print("Top-level contents:", glob.glob(DATA_ROOT + "/*"))
assert dataset_ready(), "Unzipped, but no Test folder found - send me the printed contents above."

## 1. Configuration (edit only the paths if yours differ)

In [ ]:
# ---- paths: same as notebooks 01-05 --------------------------------------
DATASET_DIR       = "/content/deepfake_data"   # filled by section 0b (local disk, fast)
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/deepfake_results"   # contains models/ and history/ from 01-05
EXTRA_DIR         = os.path.join(DRIVE_RESULTS_DIR, "extra06")    # everything new goes here
for sub in ["models", "history", "preds", "robustness", "hashes", "tables", "figures"]:
    os.makedirs(os.path.join(EXTRA_DIR, sub), exist_ok=True)

# ---- same protocol as the original run ----------------------------------
MODELS         = ["resnet18", "resnet50", "efficientnet_b0", "xception"]
IMG_SIZE       = 224
BATCH_SIZE     = 32
NUM_EPOCHS     = 15
LEARNING_RATE  = 1e-4
WEIGHT_DECAY   = 1e-4          # L2 weight decay inside torch.optim.Adam (NOT AdamW), as in notebooks 01-04
PATIENCE       = 2
THRESHOLD      = 0.5
BASE_SEED      = 42            # the seed of your existing checkpoints
NUM_WORKERS_TRAIN = 0          # 0 = identical to the original training notebooks
NUM_WORKERS_EVAL  = 2          # only affects speed of inference / robustness / hashing

# ---- extra seeds for Part H (trained IN ADDITION to seed 42) ------------
EXTRA_SEEDS    = [1, 2]        # 3 seeds total. Use [1, 2, 3, 4] for 5 seeds if you have the time.
RUN_PART_H     = True          # set False to skip the long training for now

# ---- robustness severities: level 3 = the setting used in the paper -----
SEVERITIES = {
    "JPEG":   [75, 50, 25],         # quality factor
    "Blur":   [1, 2, 3],            # Gaussian sigma
    "Noise":  [5, 10, 15],          # Gaussian pixel-noise sigma
    "Resize": [0.8, 0.65, 0.5],     # down-then-up scale factor
}
N_BOOT = 1000
EXPECTED_TEST_ACC = {"resnet18": 0.9057, "resnet50": 0.9000, "efficientnet_b0": 0.8507, "xception": 0.8990}

## 2. Shared helpers (data, models, metrics)

In [ ]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ---------- locate Train / Validation / Test folders (same logic as 01-05) ----------
def find_split_root(base_dir):
    target = {"train", "validation", "valid", "test"}
    for root, dirs, _ in os.walk(base_dir):
        if len({d.lower() for d in dirs} & target) >= 2:
            return root
    return base_dir

def resolve(root, names):
    for n in names:
        p = os.path.join(root, n)
        if os.path.isdir(p):
            return p
    return None

assert os.path.exists(DATASET_DIR), f"DATASET_DIR not found: {DATASET_DIR}"
SPLIT_ROOT = find_split_root(DATASET_DIR)
SPLIT_DIRS = {
    "train": resolve(SPLIT_ROOT, ["Train", "train"]),
    "val":   resolve(SPLIT_ROOT, ["Validation", "validation", "Valid", "valid", "Val", "val"]),
    "test":  resolve(SPLIT_ROOT, ["Test", "test"]),
}
CLASS_DIRS = {s: (resolve(d, ["Real", "real", "REAL"]), resolve(d, ["Fake", "fake", "FAKE"])) for s, d in SPLIT_DIRS.items()}
print(SPLIT_DIRS)

EXTS = ("*.jpg", "*.jpeg", "*.png", "*.bmp", "*.JPG", "*.JPEG", "*.PNG")
def list_images(d):
    files = []
    for e in EXTS:
        files += glob.glob(os.path.join(d, e))
    return sorted(set(files))

FILES = {s: {"real": list_images(r), "fake": list_images(f)} for s, (r, f) in CLASS_DIRS.items()}

def make_samples(split, seed=42):
    s = [(p, 0) for p in FILES[split]["real"]] + [(p, 1) for p in FILES[split]["fake"]]
    random.Random(seed).shuffle(s)
    return s

# ---------- transforms: identical to the original notebooks ----------
IMAGENET_MEAN = [0.485, 0.456, 0.406]; IMAGENET_STD = [0.229, 0.224, 0.225]
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

class ImgDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples, self.transform = samples, transform
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        path, label = self.samples[i]
        return self.transform(Image.open(path).convert("RGB")), torch.tensor(label, dtype=torch.float32), path

def make_loader(samples, transform, shuffle=False, workers=NUM_WORKERS_EVAL, bs=BATCH_SIZE, gen=None):
    return DataLoader(ImgDataset(samples, transform), batch_size=bs, shuffle=shuffle,
                      num_workers=workers, generator=gen)

# ---------- models: identical to the original notebooks (pretrained flag added) ----------
class BinaryClassifier(nn.Module):
    def __init__(self, backbone, target_layer):
        super().__init__(); self.backbone = backbone; self._target_layer = target_layer
    def forward(self, x): return self.backbone(x).squeeze(-1)
    def get_target_layer(self): return self._target_layer

def build_resnet18(pretrained=True):
    m = tvm.resnet18(weights=tvm.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
    m.fc = nn.Linear(m.fc.in_features, 1); return BinaryClassifier(m, m.layer4[-1])
def build_resnet50(pretrained=True):
    m = tvm.resnet50(weights=tvm.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None)
    m.fc = nn.Linear(m.fc.in_features, 1); return BinaryClassifier(m, m.layer4[-1])
def build_efficientnet_b0(pretrained=True):
    m = tvm.efficientnet_b0(weights=tvm.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, 1); return BinaryClassifier(m, m.features[-1])
def build_xception(pretrained=True):
    m = timm.create_model("xception", pretrained=pretrained, num_classes=1)
    target = None
    for _, mod in m.named_modules():
        if isinstance(mod, nn.Conv2d): target = mod
    return BinaryClassifier(m, target)
MODEL_BUILDERS = {"resnet18": build_resnet18, "resnet50": build_resnet50,
                  "efficientnet_b0": build_efficientnet_b0, "xception": build_xception}

def ckpt_path(name, seed):
    if seed == BASE_SEED:
        return os.path.join(DRIVE_RESULTS_DIR, "models", f"{name}_best.pth")
    return os.path.join(EXTRA_DIR, "models", f"{name}_seed{seed}_best.pth")

def load_model(name, seed=BASE_SEED):
    m = MODEL_BUILDERS[name](pretrained=False).to(DEVICE)
    m.load_state_dict(torch.load(ckpt_path(name, seed), map_location=DEVICE))
    return m.eval()

def save_table(df, name):
    p = os.path.join(EXTRA_DIR, "tables", name + ".csv")
    df.to_csv(p, index=False); print("saved ->", p)

In [ ]:
# ---------- metrics (pure numpy / sklearn) ----------
from sklearn.metrics import (confusion_matrix, roc_auc_score, average_precision_score,
                             brier_score_loss, f1_score)

def metrics_at(y, p, thr=0.5):
    y = np.asarray(y).astype(int); pred = (np.asarray(p) >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    prec = tp / (tp + fp) if tp + fp else float("nan")
    rec = tp / (tp + fn) if tp + fn else float("nan")
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else float("nan")
    return {"accuracy": (tp + tn) / len(y), "precision": prec, "recall": rec, "f1": f1,
            "specificity": tn / (tn + fp) if tn + fp else float("nan"),
            "fpr": fp / (fp + tn) if fp + tn else float("nan"),
            "fnr": fn / (fn + tp) if fn + tp else float("nan"),
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}

def full_metrics(y, p, thr=0.5):
    m = metrics_at(y, p, thr)
    m["roc_auc"] = roc_auc_score(y, p); m["avg_precision"] = average_precision_score(y, p)
    m["brier"] = brier_score_loss(y, p); m["ece"] = ece_score(y, p, thr)
    return m

def ece_score(y, p, thr=0.5, n_bins=10):
    y = np.asarray(y).astype(int); p = np.asarray(p)
    pred = (p >= thr).astype(int); conf = np.where(pred == 1, p, 1 - p); correct = (pred == y)
    edges = np.linspace(0, 1, n_bins + 1); ece = 0.0
    for b in range(n_bins):
        lo, hi = edges[b], edges[b + 1]
        m = (conf > lo) & (conf <= hi) if b > 0 else (conf >= lo) & (conf <= hi)
        if m.sum(): ece += m.sum() / len(conf) * abs(correct[m].mean() - conf[m].mean())
    return ece

# ---------- vectorised bootstrap helpers ----------
def boot_indices(n, B=N_BOOT, seed=0):
    return np.random.default_rng(seed).integers(0, n, size=(B, n))

def boot_acc_f1(y, pred, idx):
    yi = np.asarray(y).astype(np.int8)[idx]; pi = np.asarray(pred).astype(np.int8)[idx]
    tp = ((yi == 1) & (pi == 1)).sum(1); fp = ((yi == 0) & (pi == 1)).sum(1); fn = ((yi == 1) & (pi == 0)).sum(1)
    acc = (yi == pi).mean(1)
    f1 = np.where(2 * tp + fp + fn > 0, 2 * tp / np.maximum(2 * tp + fp + fn, 1), np.nan)
    return acc, f1

def ci(a, lo=2.5, hi=97.5):
    a = np.asarray(a); return float(np.nanmean(a)), float(np.nanpercentile(a, lo)), float(np.nanpercentile(a, hi))

def fmt_ci(t, nd=4):
    return f"{t[0]:.{nd}f} [{t[1]:.{nd}f}, {t[2]:.{nd}f}]"

# ---------- Holm-Bonferroni (no external dependency) ----------
def holm_adjust(pvals):
    p = np.asarray(pvals, dtype=float); m = len(p); order = np.argsort(p)
    adj = np.empty(m); running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (m - rank) * p[i]); adj[i] = min(1.0, running)
    return adj

## PART A — Dataset sizes (answers "where do I find train / val sizes?")
This just counts the files in your Drive copy of the dataset. **Send me the table it prints.**

In [ ]:
rows = []
for s in ["train", "val", "test"]:
    nr, nf = len(FILES[s]["real"]), len(FILES[s]["fake"])
    rows.append({"split": s, "real": nr, "fake": nf, "total": nr + nf, "pct_real": round(100 * nr / (nr + nf), 2)})
sizes_df = pd.DataFrame(rows); display(sizes_df)
print(f"Train samples: {sizes_df.total[0]} | Val samples: {sizes_df.total[1]} | Test samples: {sizes_df.total[2]}")
n_real_tr, n_fake_tr = len(FILES['train']['real']), len(FILES['train']['fake'])
ratio = max(n_real_tr, n_fake_tr) / max(1, min(n_real_tr, n_fake_tr))
POS_WEIGHT = n_real_tr / max(1, n_fake_tr) if ratio > 1.5 else 1.0
print(f"Train imbalance ratio {ratio:.3f} -> pos_weight used by the training code = {POS_WEIGHT:.3f}"
      f" ({'weighted' if ratio > 1.5 else 'UNWEIGHTED'} loss)")
save_table(sizes_df, "A_dataset_sizes")
TEST_SAMPLES = make_samples("test"); VAL_SAMPLES = make_samples("val")
Y_TEST = np.array([l for _, l in TEST_SAMPLES]); Y_VAL = np.array([l for _, l in VAL_SAMPLES])

## PART B — Baseline re-evaluation, bootstrap CIs, val-vs-test gap, threshold tuning
Loads your seed-42 checkpoints, caches validation and test probabilities, and checks that the test accuracy reproduces the paper (Table IV).

In [ ]:
@torch.no_grad()
def predict(model, samples):
    loader = make_loader(samples, eval_transform, shuffle=False)
    out = []
    for imgs, _, _ in loader:
        out.append(torch.sigmoid(model(imgs.to(DEVICE)).float()).cpu().numpy())
    return np.concatenate(out)

def cached_probs(name, seed, split):
    f = os.path.join(EXTRA_DIR, "preds", f"{name}_seed{seed}_{split}.npy")
    if os.path.exists(f): return np.load(f)
    model = load_model(name, seed)
    p = predict(model, TEST_SAMPLES if split == "test" else VAL_SAMPLES)
    np.save(f, p); del model; torch.cuda.empty_cache(); return p

P_TEST, P_VAL = {}, {}
for name in MODELS:
    P_TEST[name] = cached_probs(name, BASE_SEED, "test")
    P_VAL[name]  = cached_probs(name, BASE_SEED, "val")
    acc = metrics_at(Y_TEST, P_TEST[name])["accuracy"]
    flag = "OK" if abs(acc - EXPECTED_TEST_ACC[name]) < 0.003 else "!! DIFFERS FROM PAPER - tell me"
    print(f"{name:16s} test acc {acc:.4f} (paper {EXPECTED_TEST_ACC[name]:.4f}) {flag}")

In [ ]:
# ---- B1: bootstrap 95% CIs on the test set (seed-42 models) ----
n = len(Y_TEST); IDX = boot_indices(n, N_BOOT, seed=0)
rows, boots = [], {}
for name in MODELS:
    pred = (P_TEST[name] >= THRESHOLD).astype(int)
    acc_b, f1_b = boot_acc_f1(Y_TEST, pred, IDX)
    auc_b = np.array([roc_auc_score(Y_TEST[i], P_TEST[name][i]) for i in IDX])
    boots[name] = {"acc": acc_b, "f1": f1_b, "auc": auc_b}
    m = full_metrics(Y_TEST, P_TEST[name])
    rows.append({"model": name, "accuracy": m["accuracy"], "acc_CI": fmt_ci(ci(acc_b)),
                 "f1": m["f1"], "f1_CI": fmt_ci(ci(f1_b)), "roc_auc": m["roc_auc"], "auc_CI": fmt_ci(ci(auc_b)),
                 "ece": m["ece"], "brier": m["brier"]})
boot_df = pd.DataFrame(rows); display(boot_df); save_table(boot_df, "B1_bootstrap_ci")

# paired bootstrap differences vs ResNet-18 (same resamples)
rows = []
for name in MODELS[1:]:
    for key in ["acc", "f1", "auc"]:
        d = boots["resnet18"][key] - boots[name][key]
        t = ci(d)
        rows.append({"comparison": f"resnet18 - {name}", "metric": key, "diff_mean": t[0], "CI_low": t[1], "CI_high": t[2],
                     "CI_excludes_0": bool(t[1] > 0 or t[2] < 0)})
pd_df = pd.DataFrame(rows); display(pd_df); save_table(pd_df, "B2_paired_bootstrap_diffs")

In [ ]:
# ---- B3: validation vs test gap (manuscript B1) ----
rows = []
for name in MODELS:
    v = metrics_at(Y_VAL, P_VAL[name]); t = metrics_at(Y_TEST, P_TEST[name])
    rows.append({"model": name, "val_acc": v["accuracy"], "test_acc": t["accuracy"], "gap_pts": 100 * (v["accuracy"] - t["accuracy"]),
                 "val_specificity": v["specificity"], "test_specificity": t["specificity"],
                 "val_recall": v["recall"], "test_recall": t["recall"],
                 "val_fpr": v["fpr"], "test_fpr": t["fpr"]})
gap_df = pd.DataFrame(rows); display(gap_df); save_table(gap_df, "B3_val_vs_test_gap")
print("Where the gap comes from: compare val_specificity vs test_specificity (real images) and val_recall vs test_recall (fake images).")

In [ ]:
# ---- B4: per-model threshold tuning on VALIDATION, evaluated on TEST ----
grid = np.linspace(0.01, 0.99, 99)
def best_thr(y, p, kind):
    scores = []
    for t in grid:
        m = metrics_at(y, p, t)
        scores.append(m["f1"] if kind == "f1" else (m["recall"] - m["fpr"]))   # F1 or Youden's J
    scores = np.nan_to_num(np.array(scores), nan=-1); return float(grid[int(np.argmax(scores))])

rows = []
for name in MODELS:
    for kind, label in [("fixed", "tau=0.5"), ("f1", "val-F1-optimal"), ("youden", "val-Youden-optimal")]:
        t = 0.5 if kind == "fixed" else best_thr(Y_VAL, P_VAL[name], kind)
        m = metrics_at(Y_TEST, P_TEST[name], t)
        rows.append({"model": name, "rule": label, "threshold": t, "test_accuracy": m["accuracy"], "test_f1": m["f1"],
                     "test_recall": m["recall"], "test_specificity": m["specificity"], "test_fpr": m["fpr"], "test_fnr": m["fnr"],
                     "fp": m["fp"], "fn": m["fn"]})
thr_df = pd.DataFrame(rows); display(thr_df); save_table(thr_df, "B4_threshold_tuning")

## PART C — McNemar with multiple-comparison correction, disagreement / oracle / ensembles

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar
PRED = {m: (P_TEST[m] >= THRESHOLD).astype(int) for m in MODELS}
CORR = {m: (PRED[m] == Y_TEST) for m in MODELS}

def mcnemar_pair(a, b):
    only_a = int(np.sum(CORR[a] & ~CORR[b])); only_b = int(np.sum(~CORR[a] & CORR[b]))
    both = int(np.sum(CORR[a] & CORR[b])); neither = int(np.sum(~CORR[a] & ~CORR[b]))
    res = mcnemar([[both, only_a], [only_b, neither]], exact=(only_a + only_b) < 25, correction=True)
    return only_a, only_b, float(res.statistic), float(res.pvalue)

pairs = [(a, b) for i, a in enumerate(MODELS) for b in MODELS[i + 1:]]
rows = []
for a, b in pairs:
    oa, ob, stat, p = mcnemar_pair(a, b)
    rows.append({"pair": f"{a} vs {b}", "only_first_correct": oa, "only_second_correct": ob, "statistic": stat, "p_raw": p})
mc_df = pd.DataFrame(rows)
mc_df["p_holm_all6"] = holm_adjust(mc_df["p_raw"].values)
ref = mc_df["pair"].str.startswith("resnet18 vs")
mc_df["p_holm_vs_resnet18"] = np.nan
mc_df.loc[ref, "p_holm_vs_resnet18"] = holm_adjust(mc_df.loc[ref, "p_raw"].values)
mc_df["significant_holm_all6"] = mc_df["p_holm_all6"] < 0.05
display(mc_df); save_table(mc_df, "C1_mcnemar_holm")

In [ ]:
# ---- disagreement, oracle and simple ensembles ----
C = np.stack([CORR[m] for m in MODELS], 1)            # n x 4
rows = []
for a, b in pairs:
    rows.append({"pair": f"{a} vs {b}", "disagreement_rate": float(np.mean(CORR[a] != CORR[b])),
                 "both_wrong_rate": float(np.mean(~CORR[a] & ~CORR[b])),
                 "oracle_acc_pair": float(np.mean(CORR[a] | CORR[b]))})
dis_df = pd.DataFrame(rows); display(dis_df); save_table(dis_df, "C2_pairwise_disagreement")

n_wrong = (~C).sum(1)
print("Errors made by exactly k of the 4 models (counts of test images):", {k: int((n_wrong == k).sum()) for k in range(5)})
ens_rows = [{"system": "oracle (any of 4 correct)", "accuracy": float(np.mean(C.any(1)))}]
for m in MODELS: ens_rows.append({"system": m, "accuracy": float(np.mean(CORR[m]))})
mean4 = np.mean([P_TEST[m] for m in MODELS], 0); m4 = metrics_at(Y_TEST, mean4)
ens_rows.append({"system": "mean-prob ensemble of 4", "accuracy": m4["accuracy"], "f1": m4["f1"], "recall": m4["recall"], "specificity": m4["specificity"]})
mean3 = np.mean([P_TEST[m] for m in ["resnet18", "resnet50", "xception"]], 0); m3 = metrics_at(Y_TEST, mean3)
ens_rows.append({"system": "mean-prob ensemble R18+R50+Xception", "accuracy": m3["accuracy"], "f1": m3["f1"], "recall": m3["recall"], "specificity": m3["specificity"]})
ens_df = pd.DataFrame(ens_rows); display(ens_df); save_table(ens_df, "C3_oracle_and_ensembles")

## PART D — Robustness on the full test set (3 severities per degradation)
Level 3 of each degradation equals the setting used in the paper (JPEG q=25, blur σ=3, noise σ=15, resize ×0.5), so the numbers are directly comparable.
Noise is seeded per image so results are reproducible. Predictions are cached per condition (resumable).

In [ ]:
def deg_jpeg(img, q):
    buf = io.BytesIO(); img.save(buf, format="JPEG", quality=int(q)); buf.seek(0); return Image.open(buf).convert("RGB")
def deg_blur(img, s):
    return Image.fromarray(cv2.GaussianBlur(np.array(img), (0, 0), sigmaX=float(s)))
def deg_noise(img, s, rng):
    arr = np.array(img).astype(np.float32)
    return Image.fromarray(np.clip(arr + rng.normal(0, s, arr.shape), 0, 255).astype(np.uint8))
def deg_resize(img, f):
    w, h = img.size; return img.resize((max(1, int(w * f)), max(1, int(h * f)))).resize((w, h))

class DegradedDataset(Dataset):
    def __init__(self, samples, deg, sev):
        self.samples, self.deg, self.sev = samples, deg, sev
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        path, label = self.samples[i]; img = Image.open(path).convert("RGB")
        if self.deg == "JPEG": img = deg_jpeg(img, self.sev)
        elif self.deg == "Blur": img = deg_blur(img, self.sev)
        elif self.deg == "Noise": img = deg_noise(img, self.sev, np.random.default_rng(1_000_003 * BASE_SEED + i))
        elif self.deg == "Resize": img = deg_resize(img, self.sev)
        return eval_transform(img), torch.tensor(label, dtype=torch.float32)

BASE_MODELS = {name: load_model(name, BASE_SEED) for name in MODELS}

@torch.no_grad()
def run_condition(deg, sev):
    f = os.path.join(EXTRA_DIR, "robustness", f"{deg}_{sev}.npy")
    if os.path.exists(f): return np.load(f)
    loader = DataLoader(DegradedDataset(TEST_SAMPLES, deg, sev), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS_EVAL)
    outs = {m: [] for m in MODELS}
    for imgs, _ in tqdm(loader, desc=f"{deg}={sev}", leave=False):
        imgs = imgs.to(DEVICE)
        for m in MODELS:
            outs[m].append(torch.sigmoid(BASE_MODELS[m](imgs).float()).cpu().numpy())
    arr = np.stack([np.concatenate(outs[m]) for m in MODELS], 0); np.save(f, arr); return arr

ROB = {}
for deg, levels in SEVERITIES.items():
    for lvl, sev in enumerate(levels, 1):
        ROB[(deg, lvl)] = run_condition(deg, sev)
print("done:", len(ROB), "conditions")

In [ ]:
IDX_R = boot_indices(len(Y_TEST), 500, seed=1)
rows = []
for mi, name in enumerate(MODELS):
    clean_pred = (P_TEST[name] >= THRESHOLD).astype(int)
    f1_clean = metrics_at(Y_TEST, P_TEST[name])["f1"]
    _, f1c_b = boot_acc_f1(Y_TEST, clean_pred, IDX_R)
    for (deg, lvl), arr in ROB.items():
        p = arr[mi]; pred = (p >= THRESHOLD).astype(int)
        m = metrics_at(Y_TEST, p)
        _, f1d_b = boot_acc_f1(Y_TEST, pred, IDX_R)
        drop_b = 100 * (f1c_b - f1d_b) / f1c_b
        t = ci(drop_b)
        rows.append({"model": name, "degradation": deg, "level": lvl, "param": SEVERITIES[deg][lvl - 1],
                     "clean_f1": f1_clean, "f1": m["f1"], "accuracy": m["accuracy"],
                     "f1_drop_pct": 100 * (f1_clean - m["f1"]) / f1_clean, "drop_CI_low": t[1], "drop_CI_high": t[2]})
rob_df = pd.DataFrame(rows); save_table(rob_df, "D1_robustness_full_testset")
print("F1 drop (%) at level 3 (= paper setting) on the FULL test set:")
display(rob_df[rob_df.level == 3].pivot(index="model", columns="degradation", values="f1_drop_pct").round(2))
print("Mean F1 drop (%) over all 12 conditions (lower = more robust):")
display(rob_df.groupby("model").f1_drop_pct.mean().round(2).sort_values())
print("Full table (all levels):")
display(rob_df.pivot_table(index=["degradation", "level"], columns="model", values="f1_drop_pct").round(2))

## PART E — Duplicate / near-duplicate check
Exact duplicates (MD5 of file bytes) and near-duplicates (64-bit perceptual hash, Hamming distance) between test and train/val, and val and train.
Then re-evaluates the four models on the test images that have **no** near-duplicate in train/val.

In [ ]:
import imagehash
from concurrent.futures import ThreadPoolExecutor

def hash_one(path):
    with open(path, "rb") as f: data = f.read()
    h = imagehash.phash(Image.open(io.BytesIO(data)).convert("RGB"), hash_size=8)
    return hashlib.md5(data).hexdigest(), int(str(h), 16)

def split_hashes(split):
    f = os.path.join(EXTRA_DIR, "hashes", f"{split}.npz")
    paths = FILES[split]["real"] + FILES[split]["fake"]
    labels = np.array([0] * len(FILES[split]["real"]) + [1] * len(FILES[split]["fake"]))
    if os.path.exists(f):
        z = np.load(f, allow_pickle=True); return paths, labels, z["md5"], z["phash"]
    with ThreadPoolExecutor(max_workers=8) as ex:
        res = list(tqdm(ex.map(hash_one, paths), total=len(paths), desc=f"hash {split}"))
    md5 = np.array([r[0] for r in res]); ph = np.array([r[1] for r in res], dtype=np.uint64)
    np.savez(f, md5=md5, phash=ph); return paths, labels, md5, ph

H = {s: split_hashes(s) for s in ["train", "val", "test"]}

In [ ]:
POP = torch.tensor([bin(i).count("1") for i in range(65536)], dtype=torch.uint8, device=DEVICE)
def popcount64(x):
    return (POP[(x & 0xFFFF)].int() + POP[((x >> 16) & 0xFFFF)].int() +
            POP[((x >> 32) & 0xFFFF)].int() + POP[((x >> 48) & 0xFFFF)].int())

def min_hamming(q, r, chunk=128):
    q = torch.as_tensor(q.view(np.int64)).to(DEVICE); r = torch.as_tensor(r.view(np.int64)).to(DEVICE)
    dmin, arg = [], []
    for i in range(0, len(q), chunk):
        d = popcount64(q[i:i + chunk, None] ^ r[None, :])
        v, a = d.min(dim=1); dmin.append(v.cpu()); arg.append(a.cpu())
    return torch.cat(dmin).numpy(), torch.cat(arg).numpy()

tr_paths, tr_lab, tr_md5, tr_ph = H["train"]; va_paths, va_lab, va_md5, va_ph = H["val"]; te_paths, te_lab, te_md5, te_ph = H["test"]
d_te_tr, a_te_tr = min_hamming(te_ph, tr_ph); d_te_va, a_te_va = min_hamming(te_ph, va_ph); d_va_tr, a_va_tr = min_hamming(va_ph, tr_ph)

rows = []
for label, d, nq in [("test -> train", d_te_tr, len(te_ph)), ("test -> val", d_te_va, len(te_ph)), ("val -> train", d_va_tr, len(va_ph))]:
    row = {"query -> reference": label, "n_queries": nq}
    for t in [0, 2, 4, 6, 8]: row[f"pHash dist<={t}"] = int((d <= t).sum())
    rows.append(row)
dup_df = pd.DataFrame(rows); display(dup_df); save_table(dup_df, "E1_phash_near_duplicates")

def md5_overlap(a, b): return len(set(a) & set(b))
md5_df = pd.DataFrame([{"pair": "test & train", "identical_files": md5_overlap(te_md5, tr_md5)},
                       {"pair": "test & val", "identical_files": md5_overlap(te_md5, va_md5)},
                       {"pair": "val & train", "identical_files": md5_overlap(va_md5, tr_md5)}])
display(md5_df); save_table(md5_df, "E2_exact_duplicates_md5")

# near-duplicate pairs with CONFLICTING labels (same picture labelled real in one split, fake in another)
near = d_te_tr <= 4
conflict = int((tr_lab[a_te_tr[near]] != te_lab[near]).sum()) if near.any() else 0
print(f"Test images with a train near-duplicate (dist<=4): {int(near.sum())}; of these with a DIFFERENT label: {conflict}")

In [ ]:
# ---- re-evaluate on the test images WITHOUT near-duplicates in train or val ----
THR_NEAR = 4
tr_md5_set, va_md5_set = set(tr_md5), set(va_md5)
flag_by_path = {}
for i, p in enumerate(te_paths):
    flag_by_path[p] = bool((d_te_tr[i] <= THR_NEAR) or (d_te_va[i] <= THR_NEAR) or (te_md5[i] in tr_md5_set) or (te_md5[i] in va_md5_set))
keep = np.array([not flag_by_path[p] for p, _ in TEST_SAMPLES])
print(f"Test images kept after removing near-duplicates (dist<={THR_NEAR}) of train/val: {keep.sum()} / {len(keep)} (removed {(~keep).sum()})")
rows = []
for name in MODELS:
    full = metrics_at(Y_TEST, P_TEST[name]); clean = metrics_at(Y_TEST[keep], P_TEST[name][keep])
    rows.append({"model": name, "n_removed": int((~keep).sum()), "acc_full": full["accuracy"], "acc_dedup": clean["accuracy"],
                 "f1_full": full["f1"], "f1_dedup": clean["f1"],
                 "auc_full": roc_auc_score(Y_TEST, P_TEST[name]), "auc_dedup": roc_auc_score(Y_TEST[keep], P_TEST[name][keep])})
dedup_df = pd.DataFrame(rows); display(dedup_df); save_table(dedup_df, "E3_dedup_test_metrics")
flag_df = pd.DataFrame({"test_path": te_paths, "min_dist_to_train": d_te_tr, "min_dist_to_val": d_te_va})
flag_df = flag_df[(flag_df.min_dist_to_train <= 8) | (flag_df.min_dist_to_val <= 8)]
save_table(flag_df, "E4_flagged_test_images")

## PART F — Inference timing protocol
Synthetic 224×224 input already on the GPU (data loading and pre-processing are **excluded**), `model.eval()`, `torch.no_grad()`,
30 warm-up iterations, `torch.cuda.synchronize()` before and after every timed call, FP32 and AMP, batch size 1 and 32.

In [ ]:
@torch.no_grad()
def time_model(model, bs, amp, warmup=30, iters=None):
    iters = iters or (300 if bs == 1 else 100)
    x = torch.randn(bs, 3, IMG_SIZE, IMG_SIZE, device=DEVICE); model.eval(); ts = []
    for i in range(warmup + iters):
        torch.cuda.synchronize(); t0 = time.perf_counter()
        with torch.autocast(device_type="cuda", enabled=amp): _ = model(x)
        torch.cuda.synchronize(); dt = time.perf_counter() - t0
        if i >= warmup: ts.append(dt)
    ts = np.array(ts) * 1000
    return ts.mean() / bs, ts.std() / bs, np.median(ts) / bs

rows = []
for name in MODELS:
    for bs in [1, 32]:
        for amp in [False, True]:
            mean, sd, med = time_model(BASE_MODELS[name], bs, amp)
            rows.append({"model": name, "batch": bs, "precision": "AMP" if amp else "FP32",
                         "ms_per_img_mean": mean, "ms_per_img_sd": sd, "ms_per_img_median": med, "img_per_s": 1000 / mean})
timing_df = pd.DataFrame(rows); display(timing_df.round(3)); save_table(timing_df, "F1_timing")

## PART G — Quantitative Grad-CAM concentration
The test images are already face crops, so "focuses on the face" cannot be separated from "focuses on the centre".
This part measures how much Grad-CAM mass falls in the central 112×112 box (a uniform map would give 0.25) and in the outer 28-px border,
on 200 random test images per model, split by true class and by correct / incorrect.

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model; self.activations = None; self.gradients = None
        self.h1 = target_layer.register_forward_hook(lambda m, i, o: setattr(self, "activations", o.detach()))
        self.h2 = target_layer.register_full_backward_hook(lambda m, gi, go: setattr(self, "gradients", go[0].detach()))
    def __call__(self, x):
        self.model.zero_grad(); logit = self.model(x); logit.backward()
        w = self.gradients[0].mean(dim=(1, 2))
        cam = torch.relu((w[:, None, None] * self.activations[0]).sum(0)).cpu().numpy()
        return cam, torch.sigmoid(logit).item()
    def remove(self): self.h1.remove(); self.h2.remove()

def cam_stats(cam):
    cam = cv2.resize(cam, (IMG_SIZE, IMG_SIZE)); tot = cam.sum()
    if tot <= 0: return None
    c = cam[56:168, 56:168].sum() / tot
    border = 1 - cam[28:-28, 28:-28].sum() / tot
    return c, border

rng = np.random.default_rng(0); sel = rng.choice(len(TEST_SAMPLES), size=200, replace=False)
rows = []
for name in MODELS:
    model = BASE_MODELS[name]
    for i in tqdm(sel, desc=name, leave=False):
        path, label = TEST_SAMPLES[i]
        x = eval_transform(Image.open(path).convert("RGB")).unsqueeze(0).to(DEVICE).requires_grad_(True)
        engine = GradCAM(model, model.get_target_layer()); cam, p = engine(x); engine.remove()
        s = cam_stats(cam)
        if s is None: continue
        rows.append({"model": name, "true_class": "fake" if label else "real", "correct": int((p >= THRESHOLD) == label),
                     "center_mass": s[0], "border_mass": s[1]})
cam_df = pd.DataFrame(rows)
g = cam_df.groupby("model").agg(n=("center_mass", "size"), center_mass_mean=("center_mass", "mean"), center_mass_sd=("center_mass", "std"),
                                border_mass_mean=("border_mass", "mean")).round(3)
display(g)
g2 = cam_df.groupby(["model", "true_class", "correct"]).center_mass.agg(["size", "mean"]).round(3); display(g2)
save_table(g.reset_index(), "G1_gradcam_concentration"); save_table(g2.reset_index(), "G2_gradcam_by_class_and_correctness")
print("Uniform-map baseline for the central box = 0.25. Values well above 0.25 mean the map is centre-concentrated.")

## PART H — Extra seeds (long: about as long as notebooks 01–04 **per extra seed**)
Same protocol as the original run (Adam + L2 weight decay 1e-4, lr 1e-4, batch 32, patience 2, AMP, best-val-loss checkpoint restored).
The seed controls the new classification-head initialisation, batch order and augmentation randomness.
A `last` checkpoint is written every epoch, so if Colab disconnects just re-run this part and it resumes.
Seed 42 is **not** re-trained — your existing checkpoints are used as the first seed.

In [ ]:
USE_AMP = torch.cuda.is_available()
train_samples = make_samples("train")
train_ds = ImgDataset(train_samples, train_transform)
val_loader_h = make_loader(VAL_SAMPLES, eval_transform, shuffle=False)

def run_epoch(model, loader, criterion, optimizer=None, scaler=None):
    train = optimizer is not None; model.train() if train else model.eval()
    tot, correct, n = 0.0, 0, 0
    with (torch.enable_grad() if train else torch.no_grad()):
        for imgs, labels, _ in tqdm(loader, leave=False):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            if train: optimizer.zero_grad()
            with torch.autocast(device_type="cuda", enabled=USE_AMP):
                logits = model(imgs); loss = criterion(logits, labels)
            if train:
                scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            tot += loss.item() * imgs.size(0)
            correct += ((torch.sigmoid(logits.float()) >= THRESHOLD).float() == labels).sum().item(); n += imgs.size(0)
    return tot / n, correct / n

def train_one(name, seed):
    mdir = os.path.join(EXTRA_DIR, "models"); hist_path = os.path.join(EXTRA_DIR, "history", f"{name}_seed{seed}_history.json")
    best_p = ckpt_path(name, seed); last_p = os.path.join(mdir, f"{name}_seed{seed}_last.pth")
    if os.path.exists(hist_path) and json.load(open(hist_path)).get("finished"):
        print(f"[skip] {name} seed {seed} already finished"); return
    print(f"\n===== {name} seed {seed} ====="); set_seed(seed)
    model = MODEL_BUILDERS[name]().to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([POS_WEIGHT], device=DEVICE))
    opt = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    g = torch.Generator(); g.manual_seed(seed)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS_TRAIN, generator=g)
    hist = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}; best, no_imp, start = float("inf"), 0, 1
    if os.path.exists(last_p):
        st = torch.load(last_p, map_location=DEVICE)
        model.load_state_dict(st["model"]); opt.load_state_dict(st["opt"]); scaler.load_state_dict(st["scaler"])
        hist, best, no_imp, start = st["hist"], st["best"], st["no_imp"], st["epoch"] + 1
        print(f"resuming from epoch {start}")
    stopped = None
    for epoch in range(start, NUM_EPOCHS + 1):
        t0 = time.time()
        trl, tra = run_epoch(model, train_loader, criterion, opt, scaler); vl, va = run_epoch(model, val_loader_h, criterion)
        for k, v in zip(["train_loss", "val_loss", "train_acc", "val_acc"], [trl, vl, tra, va]): hist[k].append(v)
        print(f"epoch {epoch} [{time.time()-t0:.0f}s] train_loss={trl:.4f} train_acc={tra:.4f} val_loss={vl:.4f} val_acc={va:.4f}")
        if vl < best: best, no_imp = vl, 0; torch.save(model.state_dict(), best_p)
        else: no_imp += 1
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "scaler": scaler.state_dict(),
                    "hist": hist, "best": best, "no_imp": no_imp, "epoch": epoch}, last_p)
        json.dump(hist, open(hist_path, "w"))
        if no_imp >= PATIENCE: stopped = epoch; print(f"early stopping at epoch {epoch}"); break
    hist["finished"] = True; hist["stopped_epoch"] = stopped or len(hist["val_loss"])
    hist["best_epoch"] = int(np.argmin(hist["val_loss"]) + 1); json.dump(hist, open(hist_path, "w"))
    if os.path.exists(last_p): os.remove(last_p)
    del model; torch.cuda.empty_cache()

if RUN_PART_H:
    for seed in EXTRA_SEEDS:
        for name in MODELS:
            train_one(name, seed)
else:
    print("RUN_PART_H = False -> skipping extra-seed training.")

In [ ]:
# ---- aggregate over all available seeds (seed 42 + extra) ----
rows = []
for seed in [BASE_SEED] + EXTRA_SEEDS:
    for name in MODELS:
        if not os.path.exists(ckpt_path(name, seed)): continue
        p = cached_probs(name, seed, "test"); m = full_metrics(Y_TEST, p); m.update({"model": name, "seed": seed}); rows.append(m)
seed_df = pd.DataFrame(rows)
if len(seed_df):
    save_table(seed_df, "H1_per_seed_metrics")
    cols = ["accuracy", "precision", "recall", "f1", "roc_auc", "specificity", "ece", "brier"]
    agg = seed_df.groupby("model")[cols].agg(["mean", "std"]).round(4)
    n_seeds = seed_df.groupby("model").seed.nunique(); display(n_seeds.to_frame("n_seeds")); display(agg)
    pretty = pd.DataFrame({c: seed_df.groupby("model")[c].apply(lambda s: f"{s.mean():.4f} ± {s.std(ddof=1):.4f}") for c in cols})
    display(pretty); save_table(pretty.reset_index(), "H2_mean_sd_over_seeds")
    best_per_seed = seed_df.loc[seed_df.groupby("seed").f1.idxmax(), ["seed", "model", "f1"]]
    print("Best model by F1 in each seed:"); display(best_per_seed)
else:
    print("No seed results yet.")

## PART I — Package everything to send back

In [ ]:
for f in sorted(os.listdir(os.path.join(EXTRA_DIR, "tables"))):
    print("=" * 80); print(f)
    print(pd.read_csv(os.path.join(EXTRA_DIR, "tables", f)).to_string(max_rows=60, max_cols=20))
zip_base = "/content/extra06_tables"
shutil.make_archive(zip_base, "zip", os.path.join(EXTRA_DIR, "tables"))
shutil.copy(zip_base + ".zip", os.path.join(EXTRA_DIR, "extra06_tables.zip"))
print("\nZip also saved to Drive:", os.path.join(EXTRA_DIR, "extra06_tables.zip"))
try:
    from google.colab import files; files.download(zip_base + ".zip")
except Exception as e:
    print("download manually from Drive:", e)